In [ ]:
#"""Claims Model Training (Module 3)"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score, f1_score, classification_report
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load data
df = pd.read_csv('../data/synthetic/claims_data.csv')
print(f"Loaded {len(df)} records")
df.head()

In [ ]:
# ============================================================================
# 1. FEATURE ENGINEERING
# ============================================================================

def preprocess_claims_data(df):
    """Preprocess data for claims models"""
    
    df_processed = df.copy()
    
    # Encode categorical variables
    le_vehicle_type = LabelEncoder()
    le_damage_type = LabelEncoder()
    le_severity = LabelEncoder()
    le_status = LabelEncoder()
    
    df_processed['vehicle_type_encoded'] = le_vehicle_type.fit_transform(df_processed['vehicle_type'])
    df_processed['damage_type_encoded'] = le_damage_type.fit_transform(df_processed['damage_type'])
    df_processed['severity_encoded'] = le_severity.fit_transform(df_processed['damage_severity'])
    df_processed['status_encoded'] = le_status.fit_transform(df_processed['status'])
    
    # Number of affected parts
    df_processed['num_parts'] = df_processed['affected_parts'].str.split(',').str.len()
    
    # Create severity score
    severity_map = {'minor': 1, 'moderate': 2, 'major': 3}
    df_processed['severity_score'] = df_processed['damage_severity'].map(severity_map)
    
    # Damage type complexity
    damage_complexity = {
        'scratch': 1, 'dent': 2, 'crack': 3, 
        'major_damage': 4, 'total_loss': 5
    }
    df_processed['damage_complexity'] = df_processed['damage_type'].map(damage_complexity)
    
    return df_processed, {
        'vehicle_type': le_vehicle_type,
        'damage_type': le_damage_type,
        'severity': le_severity,
        'status': le_status
    }

df_processed, encoders = preprocess_claims_data(df)

In [ ]:
# ============================================================================
# 2. SPLIT DATA BY VEHICLE TYPE
# ============================================================================

car_df = df_processed[df_processed['vehicle_type'] == 'car'].copy()
bike_df = df_processed[df_processed['vehicle_type'] == 'bike'].copy()

print(f"Car claims: {len(car_df)}")
print(f"Bike claims: {len(bike_df)}")

In [ ]:
# ============================================================================
# 3. FEATURES FOR MODELS
# ============================================================================

feature_columns = [
    'damage_type_encoded', 'severity_encoded', 'num_parts',
    'severity_score', 'damage_complexity'
]

In [ ]:
# ============================================================================
# 4. CAR CLAIMS - AMOUNT PREDICTION (REGRESSION)
# ============================================================================

def train_amount_model(df, vehicle_type):
    """Train regression model for claim amount prediction"""
    
    X = df[feature_columns]
    y = df['predicted_amount']
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    # Scale
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train
    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train_scaled, y_train)
    
    # Evaluate
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)
    
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    test_mae = mean_absolute_error(y_test, y_test_pred)
    
    print(f"\n{vehicle_type.upper()} - Amount Model:")
    print(f"  Train R²: {train_r2:.4f}")
    print(f"  Test R²: {test_r2:.4f}")
    print(f"  Test RMSE: ₹{test_rmse:.2f}")
    print(f"  Test MAE: ₹{test_mae:.2f}")
    
    return model, scaler, X_test, y_test, y_test_pred

In [ ]:
# ============================================================================
# 5. CAR CLAIMS - APPROVAL PREDICTION (CLASSIFICATION)
# ============================================================================

def train_approval_model(df, vehicle_type):
    """Train classification model for claim approval prediction"""
    
    # Create binary target: approved = 1, else = 0
    df_model = df.copy()
    df_model['approved'] = (df_model['status'] == 'approved').astype(int)
    
    X = df_model[feature_columns]
    y = df_model['approved']
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Scale
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced'
    )
    model.fit(X_train_scaled, y_train)
    
    # Evaluate
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)
    
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)
    
    print(f"\n{vehicle_type.upper()} - Approval Model:")
    print(f"  Train Accuracy: {train_acc:.4f}")
    print(f"  Test Accuracy: {test_acc:.4f}")
    print(f"  Test F1 Score: {test_f1:.4f}")
    
    return model, scaler, X_test, y_test, y_test_pred

In [ ]:

# ============================================================================
# 6. TRAIN ALL MODELS
# ============================================================================

# Car Models
car_amount_model, car_amount_scaler, _, _, _ = train_amount_model(car_df, 'car')
car_approval_model, car_approval_scaler, _, _, _ = train_approval_model(car_df, 'car')

# Bike Models
bike_amount_model, bike_amount_scaler, _, _, _ = train_amount_model(bike_df, 'bike')
bike_approval_model, bike_approval_scaler, _, _, _ = train_approval_model(bike_df, 'bike')

In [ ]:
# ============================================================================
# 7. SAVE MODELS
# ============================================================================

os.makedirs('../models', exist_ok=True)

# Car models
joblib.dump(car_amount_model, '../models/car_amount_model.pkl')
joblib.dump(car_amount_scaler, '../models/car_amount_scaler.pkl')
joblib.dump(car_approval_model, '../models/car_approval_model.pkl')
joblib.dump(car_approval_scaler, '../models/car_approval_scaler.pkl')

# Bike models
joblib.dump(bike_amount_model, '../models/bike_amount_model.pkl')
joblib.dump(bike_amount_scaler, '../models/bike_amount_scaler.pkl')
joblib.dump(bike_approval_model, '../models/bike_approval_model.pkl')
joblib.dump(bike_approval_scaler, '../models/bike_approval_scaler.pkl')

# Save encoders
joblib.dump(encoders, '../models/claims_encoders.pkl')

print("\n✅ All models saved successfully!")

In [ ]:
# ============================================================================
# 8. MODEL COMPARISON VISUALIZATION
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Feature importance for car amount model
car_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': car_amount_model.feature_importances_
}).sort_values('importance', ascending=False)

axes[0, 0].barh(car_importance['feature'], car_importance['importance'])
axes[0, 0].set_title('Car Amount Model - Feature Importance')
axes[0, 0].set_xlabel('Importance')

# Feature importance for car approval model
car_imp_approval = pd.DataFrame({
    'feature': feature_columns,
    'importance': car_approval_model.feature_importances_
}).sort_values('importance', ascending=False)

axes[0, 1].barh(car_imp_approval['feature'], car_imp_approval['importance'])
axes[0, 1].set_title('Car Approval Model - Feature Importance')
axes[0, 1].set_xlabel('Importance')

# Feature importance for bike amount model
bike_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': bike_amount_model.feature_importances_
}).sort_values('importance', ascending=False)

axes[1, 0].barh(bike_importance['feature'], bike_importance['importance'])
axes[1, 0].set_title('Bike Amount Model - Feature Importance')
axes[1, 0].set_xlabel('Importance')

# Feature importance for bike approval model
bike_imp_approval = pd.DataFrame({
    'feature': feature_columns,
    'importance': bike_approval_model.feature_importances_
}).sort_values('importance', ascending=False)

axes[1, 1].barh(bike_imp_approval['feature'], bike_imp_approval['importance'])
axes[1, 1].set_title('Bike Approval Model - Feature Importance')
axes[1, 1].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('../models/claims_model_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Model comparison saved!")

In [ ]:
# ============================================================================
# 9. TEST PREDICTION FUNCTION
# ============================================================================

def predict_claim(claim_data, models, encoders):
    """Predict claim amount and approval probability"""
    
    # Extract data
    vehicle_type = claim_data['vehicle_type']
    damage_type = claim_data['damage_type']
    severity = claim_data['damage_severity']
    affected_parts = claim_data['affected_parts']
    num_parts = len(affected_parts.split(','))
    
    # Encode
    damage_type_encoded = encoders['damage_type'].transform([damage_type])[0]
    severity_encoded = encoders['severity'].transform([severity])[0]
    
    severity_map = {'minor': 1, 'moderate': 2, 'major': 3}
    severity_score = severity_map[severity]
    
    damage_complexity = {
        'scratch': 1, 'dent': 2, 'crack': 3, 
        'major_damage': 4, 'total_loss': 5
    }
    damage_complexity_score = damage_complexity[damage_type]
    
    # Prepare features
    features = np.array([[
        damage_type_encoded,
        severity_encoded,
        num_parts,
        severity_score,
        damage_complexity_score
    ]])
    
    # Select appropriate models
    if vehicle_type == 'car':
        amount_model = models['car_amount']
        amount_scaler = models['car_amount_scaler']
        approval_model = models['car_approval']
        approval_scaler = models['car_approval_scaler']
    else:
        amount_model = models['bike_amount']
        amount_scaler = models['bike_amount_scaler']
        approval_model = models['bike_approval']
        approval_scaler = models['bike_approval_scaler']
    
    # Predict amount
    features_scaled_amount = amount_scaler.transform(features)
    predicted_amount = amount_model.predict(features_scaled_amount)[0]
    
    # Predict approval probability
    features_scaled_approval = approval_scaler.transform(features)
    approval_prob = approval_model.predict_proba(features_scaled_approval)[0][1]
    
    return {
        'predicted_amount': float(predicted_amount),
        'approval_probability': float(approval_prob)
    }

# Test prediction
test_claim = {
    'vehicle_type': 'car',
    'damage_type': 'dent',
    'damage_severity': 'moderate',
    'affected_parts': 'bumper, door'
}

models = {
    'car_amount': car_amount_model,
    'car_amount_scaler': car_amount_scaler,
    'car_approval': car_approval_model,
    'car_approval_scaler': car_approval_scaler,
    'bike_amount': bike_amount_model,
    'bike_amount_scaler': bike_amount_scaler,
    'bike_approval': bike_approval_model,
    'bike_approval_scaler': bike_approval_scaler
}

result = predict_claim(test_claim, models, encoders)
print(f"\nTest Claim Prediction:")
print(f"Vehicle: {test_claim['vehicle_type']}")
print(f"Damage: {test_claim['damage_type']} ({test_claim['damage_severity']})")
print(f"Predicted Amount: ₹{result['predicted_amount']:.2f}")
print(f"Approval Probability: {result['approval_probability']*100:.1f}%")

print("\n✅ Claims model training complete!")